# 03 — Preprocessing and Baseline Models

## Objective

This notebook builds a reproducible preprocessing pipeline and establishes baseline classification models for customer churn prediction.

The main goals are:

- define the predictive features and target;
- create an isolated training and test set;
- preprocess numerical and categorical variables;
- prevent data leakage;
- establish a simple baseline;
- train an interpretable logistic regression model;
- evaluate initial model performance.

In [1]:
from pathlib import Path

import pandas as pd

from sklearn.model_selection import train_test_split

In [2]:
PROJECT_ROOT = Path("..")
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "WA_FnUseC_TelcoCustomerChurn.csv"

df = pd.read_csv(DATA_PATH)

df.shape

(7043, 21)

In [3]:
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

## 1. Features and target

`Churn` is the target variable.

`customerID` is excluded because it uniquely identifies customers and does not represent a meaningful predictive customer characteristic.

In [4]:
X = df.drop(columns=["customerID", "Churn"])
y = df["Churn"]

In [5]:
X.shape, y.shape

((7043, 19), (7043,))

## 2. Train/test split

The test set is isolated before building or selecting models.

Model development and validation will be performed using the training data. The test set will be reserved for final evaluation.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [7]:
print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)

Training shape: (5634, 19)
Test shape: (1409, 19)


In [8]:
y_test.value_counts(normalize=True)

Churn
No     0.734564
Yes    0.265436
Name: proportion, dtype: float64

## 3. Feature preprocessing

Numerical and categorical variables require different preprocessing steps.

The preprocessing workflow will be fitted only on the training data to avoid leaking information from the test set.

In [9]:
numeric_features = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

categorical_features = [
    column
    for column in X_train.columns
    if column not in numeric_features
]

In [10]:
print("Numerical features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

Numerical features:
['tenure', 'MonthlyCharges', 'TotalCharges']

Categorical features:
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [12]:
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

In [13]:
categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

In [14]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ]
)

In [15]:
from sklearn.dummy import DummyClassifier

In [16]:
dummy_model = DummyClassifier(
    strategy="most_frequent"
)

In [17]:
dummy_model.fit(X_train, y_train)

,"strategy strategy: {""most_frequent"", ""prior"", ""stratified"", ""uniform"", ""constant""}, default=""prior""Strategy to use to generate predictions.* ""most_frequent"": the `predict` method always returns the most frequent class label in the observed `y` argument passed to `fit`. The `predict_proba` method returns the matching one-hot encoded vector.* ""prior"": the `predict` method always returns the most frequent class label in the observed `y` argument passed to `fit` (like ""most_frequent""). ``predict_proba`` always returns the empirical class distribution of `y` also known as the empirical class prior distribution.* ""stratified"": the `predict_proba` method randomly samples one-hot vectors from a multinomial distribution parametrized by the empirical class prior probabilities. The `predict` method returns the class label which got probability one in the one-hot vector of `predict_proba`. Each sampled row of both methods is therefore independent and identically distributed.* ""uniform"": generates predictions uniformly at random from the list of unique classes observed in `y`, i.e. each class has equal probability.* ""constant"": always predicts a constant label that is provided by the user. This is useful for metrics that evaluate a non-majority class. .. versionchanged:: 0.24 The default value of `strategy` has changed to ""prior"" in version 0.24.",'most_frequent'
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness to generate the predictions when``strategy='stratified'`` or ``strategy='uniform'``.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",None
,"constant constant: int or str or array-like of shape (n_outputs,), default=NoneThe explicit constant as predicted by the ""constant"" strategy. Thisparameter is useful only for the ""constant"" strategy.",None
Name,Type,Value
"class_prior_ class_prior_: ndarray of shape (n_classes,) or list of such arraysFrequency of each class observed in `y`. For multioutput classificationproblems, this is computed independently for each output.","ndarray[float64](2,)","[0.73,0.27]"
"classes_ classes_: ndarray of shape (n_classes,) or list of such arraysUnique class labels observed in `y`. For multi-output classificationproblems, this attribute is a list of arrays as each output has anindependent set of possible classes.","ndarray[object](2,)","['No','Yes']"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X` hasfeature names that are all strings.","ndarray[object](19,)","['gender','SeniorCitizen','Partner',...,'PaymentMethod','MonthlyCharges', 'TotalCharges']"
n_classes_ n_classes_: int or list of intNumber of label for each output.,int,2
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`.,int,19
n_outputs_ n_outputs_: intNumber of outputs.,int,1
sparse_output_ sparse_output_: boolTrue if the array returned from predict is to be in sparse CSC format.Is automatically set to True if the input `y` is passed in sparseformat.,bool,False


In [18]:
dummy_predictions = dummy_model.predict(X_test)

In [19]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

In [20]:
dummy_accuracy = accuracy_score(
    y_test,
    dummy_predictions
)

dummy_accuracy

0.7345635202271115

In [21]:
confusion_matrix(
    y_test,
    dummy_predictions,
    labels=["No", "Yes"]
)

array([[1035,    0],
       [ 374,    0]])

In [22]:
print(
    classification_report(
        y_test,
        dummy_predictions,
        zero_division=0
    )
)

              precision    recall  f1-score   support

          No       0.73      1.00      0.85      1035
         Yes       0.00      0.00      0.00       374

    accuracy                           0.73      1409
   macro avg       0.37      0.50      0.42      1409
weighted avg       0.54      0.73      0.62      1409



### Dummy baseline interpretation

The majority-class dummy classifier achieves relatively high accuracy because the dataset contains substantially more non-churn customers than churn customers.

However, the model fails to identify churned customers, producing a recall of zero for the positive class.

This demonstrates why accuracy alone is not an appropriate metric for evaluating churn models and establishes a minimum baseline that future models must outperform.